In [1]:
%pip install -r requirements.txt

Looking in links: https://download.pytorch.org/whl/torch_stable.html
Note: you may need to restart the kernel to use updated packages.


### Implementing Scaled Dot-Product Attention: $softmax(\frac{QK^T}{\sqrt d_k})V$


In [2]:
import torch
import torch.nn.functional as F
import math
import plotly.graph_objects as go # For visualization
import numpy as np

In [3]:
def scaled_dot_product_attention(query, key, value, mask=None):
    """
    Computes Scaled Dot-Product Attention.

    Args:
        query: Query tensor; shape (batch_size, seq_len_q, d_k)
        key: Tensor; shape (batch_size, seq_len_k, d_k)
        value: Value tensor; shape (batch_size, seq_len_k, d_v)
               Note: seq_len_k must match between key and value.
               d_v (value dimension) can differ from d_k (query dimension).
        mask: Optional mask tensor; shape broadcastable to (batch_size, seq_len_q, seq_len_k).
              Positions with True or 1 indicate values to keep, False or 0 to mask out.

    Returns:
        output: The attention output tensor; shape (batch_size, seq_len_q, d_v)
        attention_weights: The attention weights; shape (batch_size, seq_len_q, seq_len_k)
    """

    #dimension of key
    d_k = key.size(-1)

    # 1. QK^T: (batch_size, seq_len_q, d_k) @ (batch_size, d_k, seq_len_k) -> (batch_size, seq_len_q, seq_len_k)
    scores = torch.matmul(query, key.transpose(-2,-1))

    # 2. Scale by sqrt(d_k)
    scaled_scores = scores / math.sqrt(d_k)

    # 3. Apply mask if provided
    # if mask position is 0, set the scaled_scores to -inf so that the softmax of it can be 0
    if mask is not None:
        scaled_scores = scaled_scores.masked_fill(mask == 0, -1e9)

    # 4. Softmax is applied on the last dimension (seq_len_k)
    attention_weights = F.softmax(scaled_scores, dim = -1)

    # Replace NaN with 0 to handle potential NaN from softmax if all scores in a row are -inf
    # This can happen if a query position is masked entirely against all keys.
    attention_weights = torch.nan_to_num(attention_weights)

    # 5. Multiply weights by V
    # (batch_size, seq_len_q, seq_len_k) @ (batch_size, seq_len_k, d_v) -> (batch_size, seq_len_q, d_v)
    outputs = torch.matmul(attention_weights, value)
    
    return outputs, attention_weights

#### Test code

In [4]:
batch_size = 1
seq_len_q = 3 # Sequence length for queries
seq_len_k = 4 # Sequence length for keys/values
d_k = 4      # Dimension of keys/queries
d_v = 4      # Dimension of values

# Generate random tensors
query = torch.randn(batch_size, seq_len_q, d_k)
key = torch.randn(batch_size, seq_len_k, d_k)
value = torch.randn(batch_size, seq_len_k, d_v)

# no mask
output, attention_weights = scaled_dot_product_attention(query, key, value)
print("--- Output without Mask ---")
print("Output shape:", output.shape) # Expected: [1, 3, 16]
print("Attention weights shape:", attention_weights.shape) # Expected: [1, 3, 4]
# Each row in attention_weights should sum to 1
print("Attention weights sum (second query):", attention_weights[0, 1, :].sum())

# with mask
# Create a sample mask. Let's mask out the last position for all queries.
# Mask shape: (batch_size, seq_len_q, seq_len_k)
# Here, simpler: (batch_size, 1, seq_len_k) broadcastable
mask = torch.ones(batch_size, 1, seq_len_k, dtype=torch.bool)
mask[:, :, -1] = 0 # Mask the last position (index 3)
# we mask it out for all query tokens and all attention heads at last position of each query

print("\nMask shape:", mask.shape)
print("Mask content:\n", mask)

output_masked, attention_weights_masked = scaled_dot_product_attention(query, key, value, mask=mask)

print("\n--- Output with Mask ---")
print("Output shape:", output_masked.shape) # Expected: [1, 3, 16]
print("Attention weights shape:", attention_weights_masked.shape) # Expected: [1, 3, 4]
print("Masked Attention weights (first query):\n", attention_weights_masked[0, 0, :])
# Note that the weight for the last position (index 3) should be 0 or very close to it.
print("Attention weights sum (first query, masked):", attention_weights_masked[0, 0, :].sum())

--- Output without Mask ---
Output shape: torch.Size([1, 3, 4])
Attention weights shape: torch.Size([1, 3, 4])
Attention weights sum (second query): tensor(1.)

Mask shape: torch.Size([1, 1, 4])
Mask content:
 tensor([[[ True,  True,  True, False]]])

--- Output with Mask ---
Output shape: torch.Size([1, 3, 4])
Attention weights shape: torch.Size([1, 3, 4])
Masked Attention weights (first query):
 tensor([0.8844, 0.0636, 0.0520, 0.0000])
Attention weights sum (first query, masked): tensor(1.)


In [5]:
print(query)
print(key.transpose(-2,-1))
scaled_scores = torch.matmul(query, key.transpose(-2,-1))/math.sqrt(d_k)
print(scaled_scores)
print(mask)
scaled_scores = scaled_scores.masked_fill(mask == 0, -1e9)
print(scaled_scores)
print(attention_weights)
print(value)
print(output)

tensor([[[-0.4897,  2.3984,  0.8191, -0.9611],
         [-0.6701, -1.1832,  0.1919, -0.0639],
         [-0.0817,  0.4123,  0.5008,  0.3179]]])
tensor([[[ 0.1098, -0.6588, -1.1593,  0.2264],
         [ 1.7892, -0.8476, -0.7025,  0.5469],
         [-0.1387, -0.6177,  0.5988,  0.0841],
         [ 0.3006, -0.8170,  1.2551,  0.9178]]])
tensor([[[ 1.9175, -0.7155, -0.9165,  0.1938],
         [-1.1182,  0.6890,  0.8213, -0.4206],
         [ 0.3773, -0.4324,  0.2520,  0.2704]]])
tensor([[[ True,  True,  True, False]]])
tensor([[[ 1.9175e+00, -7.1551e-01, -9.1648e-01, -1.0000e+09],
         [-1.1182e+00,  6.8903e-01,  8.2133e-01, -1.0000e+09],
         [ 3.7734e-01, -4.3237e-01,  2.5200e-01, -1.0000e+09]]])
tensor([[[0.7639, 0.0549, 0.0449, 0.1363],
         [0.0623, 0.3795, 0.4332, 0.1251],
         [0.3100, 0.1379, 0.2735, 0.2786]]])
tensor([[[-0.9725,  0.4331, -0.7220,  1.0797],
         [-0.4871,  1.9296,  0.1467,  0.0537],
         [ 1.8229, -0.9611, -0.8037,  1.7335],
         [ 2.0577, -

In [63]:
# Use the attention_weights from the unmasked example
weights_np = attention_weights[0].detach().numpy() # Get weights for the first batch item

# Create heatmap data
fig_data = go.Heatmap(
    z=weights_np,
    x=[f'Pos {i}' for i in range(seq_len_k)],
    y=[f'Query Pos {i}' for i in range(seq_len_q)],
    colorscale='Blues', # Use blue color scale
    colorbar=dict(title='Attention Weight')
)

# Create layout
fig_layout = go.Layout(
    title='Attention Weights (Query vs Key)',
    xaxis_title="Sequence Position",
    yaxis_title="Query Sequence Position",
    yaxis_autorange='reversed', # Show query 0 at the top
    width=500, height=400, margin=dict(l=50, r=50, b=100, t=100, pad=4)
)

# Generate figure object as JSON for web display
fig = go.Figure(data=[fig_data], layout=fig_layout)
plotly_json = fig.to_json()

fig.show()

